# Homework 1

## Question 1

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
len(documents)

72

## Question 2

In [4]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [5]:
question = "How does the agentic loop keep calling the model until it stops?"
res = index.search(
        question,
        num_results=1
)

res[0]

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

In [6]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [7]:
len(chunks)

295

# Question 3

In [8]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-19 10:40:21--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-19 10:40:22 (20.8 MB/s) - ‘rag_helper.py’ saved [2134/2134]



In [9]:
import os
from openai import OpenAI

from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
    base_url="https://api.anthropic.com/v1/",
)

In [10]:
from homework.rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=client,
    model="claude-haiku-4-5"
)

In [12]:
answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [13]:
answer.usage.prompt_tokens

8193

## Question 4

In [14]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [15]:
len(chunks)

295

## Question 5

In [17]:
chunked_idx = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunked_idx.fit(chunks)

In [18]:
chunked_assistant = RAGBase(
    index=chunked_idx,
    llm_client=client,
    model="claude-haiku-4-5"
)

In [19]:
answer_chunked = chunked_assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [22]:
answer_chunked.usage.prompt_tokens / answer.usage.prompt_tokens

0.3233247894544123

## Question 6

In [23]:
!uv add toyaikit

Resolved 129 packages in 1.22s                                       
Prepared 7 packages in 1.03s                                             
Installed 7 packages in 19ms                                
 + anthropic==0.111.0
 + docstring-parser==0.18.0
 + genai-prices==0.0.66
 + httpcore2==2.4.0
 + httpx2==2.4.0
 + toyaikit==0.0.11
 + truststore==0.10.4


In [24]:
def search(query):
    return chunked_idx.search(
        query,
        num_results=5,
    )

In [25]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [30]:
from toyaikit import AnthropicClient, AnthropicMessagesRunner
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

In [31]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [32]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course FAQ.'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [33]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [41]:
prompt = """
You're a course teaching assistant.
Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
"""

In [49]:
llm_client = AnthropicClient(
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
    model="claude-haiku-4-5-20251001",
)

runner = AnthropicMessagesRunner(
    tools=agent_tools,
    developer_prompt=prompt,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [50]:
runner.run()

You: How does the agentic loop work, and how is it different from plain RAG?


-> Response received


-> Response received


-> Response received


Aspect,Plain RAG,Agentic Loop
Search calls,Fixed (1),Variable (model decides)
API calls,1 call to the LLM,Multiple calls (one per iteration)
Error recovery,"None - if search is bad, answer is bad",Model can retry with different search terms
Cost,Lower and predictable,Higher - scales with loop iterations
Latency,Lower - single round-trip,Higher - multiple round-trips
Flexibility,Predetermined flow,"Dynamic, adapts to results"


KeyboardInterrupt: Interrupted by user